## Example 1: Render a picture as text with Rich (unicode renderer)

In [ ]:
import io
from PIL import Image as PILImage
from rich.console import Console
from textual_image.renderable import UnicodeImage

picture = PILImage.new("RGB", (32, 16), (200, 30, 30))
console = Console(file=io.StringIO(), width=40, force_terminal=False, force_jupyter=False)
console.print(UnicodeImage(picture, width=20))
text = console.file.getvalue()
assert text.strip(), "nothing rendered"
print(text)


## Example 2: Half-cell renderer (two pixels per character, coloured)

In [ ]:
import io
from PIL import Image as PILImage
from rich.console import Console
from textual_image.renderable import HalfcellImage

picture = PILImage.linear_gradient("L").convert("RGB").resize((24, 24))
console = Console(file=io.StringIO(), width=30, force_terminal=True, force_jupyter=False, color_system="truecolor")
console.print(HalfcellImage(picture, width=12))
out = console.file.getvalue()
assert "▀" in out or "▄" in out, out[:200]
print(len(out.splitlines()), "lines rendered")


## Example 3: The automatic choice off a terminal falls back to unicode

In [ ]:
from textual_image.renderable import Image, UnicodeImage, HalfcellImage, SixelImage, TGPImage

print("chosen renderer:", Image.__module__)
assert Image in (UnicodeImage, HalfcellImage, SixelImage, TGPImage)


## Example 4: A picture from a file path and a byte stream

In [ ]:
import io, os, tempfile
from PIL import Image as PILImage
from rich.console import Console
from textual_image.renderable import UnicodeImage

path = os.path.join(tempfile.mkdtemp(), "p.png")
PILImage.new("RGB", (10, 10), (0, 120, 255)).save(path)
data = io.BytesIO(open(path, "rb").read())
for source in (path, data):
    console = Console(file=io.StringIO(), width=20, force_jupyter=False)
    console.print(UnicodeImage(source, width=8))
    assert console.file.getvalue().strip()
print("path and bytes ok")


## Example 5: The Textual widget in a headless app

In [ ]:
from PIL import Image as PILImage
from textual.app import App
from textual_image.widget import HalfcellImage

class Viewer(App):
    def compose(self):
        yield HalfcellImage(PILImage.new("RGB", (20, 20), (10, 200, 10)))

app = Viewer()
async with app.run_test(size=(40, 20)) as pilot:
    await pilot.pause()
    widget = app.query_one(HalfcellImage)
    assert widget is not None
print("widget mounted", type(widget).__name__)
